In [4]:
import pandas as pd
import numpy as np

# === config ===
#csv_path = "data/test/dataset_test_x1000.csv"
csv_path = "data/test/dataset_test_x1000_augmented.csv"
TOL = 1e-6
ORDER = ["001","010","100","011","101","110","111","000"]  # preferred print order

# === load ===
df = pd.read_csv(csv_path, header=None)

# CSV layout in your repo: ID, ρ, θ1, θ2, θ3, then 21 C-entries (Mandel)
theta_df = df.iloc[:, 2:5].astype(float)   # θ1, θ2, θ3

# === make pattern label per row ===
def to_pattern(row, tol=TOL):
    bits = ['1' if abs(v) > tol else '0' for v in row]
    return ''.join(bits)

patterns = theta_df.apply(to_pattern, axis=1)
df["theta_pattern"] = patterns

# === group indices by pattern ===
pattern_groups = {}
for idx, pat in enumerate(patterns):
    pattern_groups.setdefault(pat, []).append(idx)

# === print summary with indices ===
print(f"✅ Theta-pattern summary for {csv_path} \n")
for pat in ORDER + sorted([p for p in pattern_groups if p not in ORDER]):
    idxs = pattern_groups.get(pat, [])
    print(f"{pat}: count={len(idxs)}, indices={idxs}")

print(f"\nTotal rows: {len(df)}")


✅ Theta-pattern summary for data/test/dataset_test_x1000_augmented.csv 

001: count=333, indices=[4003, 4006, 4009, 4012, 4015, 4018, 4021, 4024, 4027, 4030, 4033, 4036, 4039, 4042, 4045, 4048, 4051, 4054, 4057, 4060, 4063, 4066, 4069, 4072, 4075, 4078, 4081, 4084, 4087, 4090, 4093, 4096, 4099, 4102, 4105, 4108, 4111, 4114, 4117, 4120, 4123, 4126, 4129, 4132, 4135, 4138, 4141, 4144, 4147, 4150, 4153, 4156, 4159, 4162, 4165, 4168, 4171, 4174, 4177, 4180, 4183, 4186, 4189, 4192, 4195, 4198, 4201, 4204, 4207, 4210, 4213, 4216, 4219, 4222, 4225, 4228, 4231, 4234, 4237, 4240, 4243, 4246, 4249, 4252, 4255, 4258, 4261, 4264, 4267, 4270, 4273, 4276, 4279, 4282, 4285, 4288, 4291, 4294, 4297, 4300, 4303, 4306, 4309, 4312, 4315, 4318, 4321, 4324, 4327, 4330, 4333, 4336, 4339, 4342, 4345, 4348, 4351, 4354, 4357, 4360, 4363, 4366, 4369, 4372, 4375, 4378, 4381, 4384, 4387, 4390, 4393, 4396, 4399, 4402, 4405, 4408, 4411, 4414, 4417, 4420, 4423, 4426, 4429, 4432, 4435, 4438, 4441, 4444, 4447, 4450, 44

In [5]:
# === 3 example C rows (21 Mandel values) per theta pattern ===
from utils.data_utils.load_data import full_C_from_C_flat_21, extract_target_properties  # (not used below, but handy)

PATTERNS = ["001","010","100","011","101","110","111"]
K = 3                 # exactly 3 per pattern
SAMPLE_MODE = "first" # "first" or "random"
SEED = 42

rng = np.random.default_rng(SEED)

def row_to_C21(row):
    # CSV layout: [ID, rho, theta1, theta2, theta3, C(21)...]
    return row.iloc[5:26].astype(float).to_numpy()

def pick_k(idxs, k=3, mode="first"):
    if not idxs:
        return []
    if mode == "random":
        take = rng.choice(idxs, size=min(k, len(idxs)), replace=False).tolist()
        take.sort()
        return take
    return idxs[:k]

def fmt_csv_line(vals):
    # compact + precise; scientific when needed
    return ",".join(f"{v:.9g}" for v in vals)

all_lines = []
labeled = []  # (pattern, idx, line)

for pat in PATTERNS:
    idxs = pattern_groups.get(pat, [])
    chosen = pick_k(idxs, k=K, mode=SAMPLE_MODE)
    if len(chosen) < K:
        print(f"⚠️ pattern {pat}: only {len(chosen)} available (requested {K})")
    for idx in chosen:
        C21 = row_to_C21(df.iloc[idx])
        line = fmt_csv_line(C21)
        all_lines.append(line)
        labeled.append((pat, idx, line))

# --- clean block to paste directly into a CSV (21 numbers per row) ---
print("\n=== COPY BELOW (21 values per row) ===")
print("\n".join(all_lines))

# --- labeled preview so you know what came from where (non-CSV) ---
print("\n=== Preview (pattern, index) ===")
for pat, idx, line in labeled:
    print(f"{pat}\tidx={idx}\t{line}")



=== COPY BELOW (21 values per row) ===
0.9178475,0.35223496,0.35432178,0.0009335876,-0.0027022988,-0.0011657763,0.9125454,0.35255313,0.0010205914,-0.00022904793,-0.0014834827,0.92141813,0.00045480428,-0.0014517991,-0.0016266232,0.55763054,-0.001534997,-0.00031241315,0.5612802,0.0005712914,0.55807084
0.24169235,0.068577334,0.039186683,-0.005655483,0.006989135,-1.0317925e-05,0.24725859,0.03878614,-0.0055699926,0.0020225972,-0.005619147,0.05129427,-2.7789276e-05,0.0027505883,-0.0016450287,0.17340596,-0.0012581145,0.004046211,0.07042271,5.69089e-06,0.069088526
0.4940061,0.14705853,0.043108314,0.0009914004,0.007912435,-0.001346031,0.506191,0.042039737,-0.0007138363,0.0014421998,-0.004814323,0.06013678,-0.0003249539,0.0012005641,-0.00025571894,0.3519364,-0.0034145445,0.0044784076,0.064274326,0.00017134364,0.0644189
0.9125454,0.35255313,0.35223496,-0.0014834827,0.0010205914,-0.00022904793,0.92141813,0.35432178,-0.0016266232,0.00045480428,-0.0014517991,0.9178475,-0.0011657763,0.0009335876,-0.